In [ ]:
# Importing Libraries
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.animation as animation
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import scipy as sp
import numpy as np
import pandas as pd
import tkinter as tk
import time
import datetime
import os
import sys
import pyvisa
import threading
import ipywidgets
import nidaqmx
import warnings
from tkinter import filedialog
from scipy import signal
from collections import deque
from IPython.display import clear_output, display
from nidaqmx.constants import AcquisitionType, TerminalConfiguration

print(f"All Libraries imported successfully at {datetime.datetime.now()}")

All Libraries imported successfully at 2026-04-24 13:19:36.296087


In [2]:
# Open Pyvisa Resource Manager and connect to insturements at 14T cryostat
rm = pyvisa.ResourceManager()

bilt = rm.open_resource("TCPIP0::192.168.150.123::5025::SOCKET", timeout=5000, write_termination='\n', read_termination='\n')
print(bilt.query("*IDN?"))

print(f"\nInstruments connected successfully at {datetime.datetime.now()}")

2142,"ITEST BE2142C/12V 15mA DC-SOURCE/SN06-023 LC2002 VL446\240"

Instruments connected successfully at 2026-04-24 13:23:35.009283


In [280]:
def measure_bilt(address,position,channel):
    Voltage = bilt.query(f"I{position};C{channel};MEAS:VOLT?")
    Current = bilt.query(f"I{position};C{channel};MEAS:CURR?")
    return Voltage, Current

def setvolt_bilt(address,position,channel,voltage,step,steptime=10):
    address.write(f"I{position};C{channel};TRIG:IN 1") # 0: Exponetial, 1: Ramp, 2: Staircase, 3: Triggered Step, 4: Auto Step
    address.write(f"I{position};C{channel};VOLT:STEP:WIDTH {steptime}") # Step width in milliseconds
    address.write(f"I{position};C{channel};VOLT:STEP:AMPL {step}") # Step size in volts
    address.write(f"I{position};C{channel};VOLT {voltage}")  # Target voltage in volts
    address.write(f"I{position};C{channel};TRIG:IN:INIT")  # Triggering

In [ ]:
for i in range(5000):
    Voltage, Current = float(measure_bilt(bilt, 3, 3)[0]), float(measure_bilt(bilt, 3, 3)[1])
    print(f"Voltage : {Voltage}; Current : {Current}; Time : {datetime.datetime.now()}", end="\r")
    time.sleep(0.5)

In [284]:
setvolt_bilt(bilt, 3, 3, voltage=0.0, step=0.01, steptime=100)

In [4]:
scope_module = session.modules.scope # Accessing the scope module.

In [295]:
dmm1 = rm.open_resource("TCPIP0::192.168.150.124::gpib0,15::INSTR", timeout=5000, write_termination='\n', read_termination='\n')
print(dmm1.query("*IDN?"))

dmm2 = rm.open_resource("TCPIP0::192.168.150.124::gpib0,25::INSTR", timeout=5000, write_termination='\n', read_termination='\n')
print(dmm2.query("*IDN?"))

HEWLETT-PACKARD,34401A,0,8-5-2
HEWLETT-PACKARD,34401A,0,11-5-2


In [305]:
print(list(session.child_nodes()))

[/zi/config, /zi/about, /zi/debug, /zi/clockbase, /zi/devices, /zi/mds]


In [14]:
uhf.clockbase()

1800000000.0

In [ ]:
scope_module = session.modules.scope # Accessing the scope module.

scope_module.mode(3) # Acquisition mode. 0 for Scope raw data, 1 for averaging mode, 3 for FFT mode.
scope_module.fft.window(1) # FFT window type: 0 > rectangular, 1 > Hann, 2 > Hamming, 3 > Blackman-Harris, 4 > Flat-Top.
scope_module.fft.powercompensation(1) # Enable power compensation for noise floor correction. 0 > no, 1 > yes.
scope_module.fft.power(1) # Enable power units (V^2) for correct noise floor units. 0 > no, 1 > yes.
scope_module.fft.spectraldensity(1) # Enable spectral density (divide by bin width) for correct noise floor units. 0 > no, 1 > yes.

scope_module.historylength(100) # Maximum number of records stored in history buffer (circular buffer).

scope_module.averager.enable(1) # Enables (1) or disables (0) the averager.
scope_module.averager.method(1) # Sets the averaging method. 0 for exponential moving average, 1 for uniform averaging.
scope_module.averager.resamplingmode(0) # Sets tne resampling mode for low sample rate signals. 0 for linear interpolation, 1 for pchip interpolation.
scope_module.averager.weight(1) # Sets the averaging weight used in exponential moving average.
scope_module.averager.restart(0) # Set to 1 to restart the averager with the next acquired record. Otherwise 0.

wave_node = uhf.scopes[0].wave # Accessing the wave node of the scope.
scope_module.subscribe(wave_node) # Subscribing to the wave node to receive data.
clockbase = uhf.clockbase() # Get the clock base frequency of the device

In [397]:
clockbase = uhf.clockbase() # Get the clock base frequency of the device

#Obtain scope records from the device using an instance of the Scope Module.

def check_scope_record_flags(scope_records, num_records):
    # Loop over all records and print a warning to the console if an error bit in flags has been set.
    num_records = len(scope_records)
    for index, record in enumerate(scope_records):
        record_idx = f"{index}/{num_records}"
        record_flags = record[0]["flags"]
        if record_flags & 1:
            print(f"Warning: Scope record {record_idx} flag indicates dataloss.")
        if record_flags & 2:
            print(f"Warning: Scope record {record_idx} indicates missed trigger.")
        if record_flags & 4:
            print(f"Warning: Scope record {record_idx} indicates transfer failure (corrupt data).")
        totalsamples = record[0]["totalsamples"]
        for wave in record[0]["wave"]:
            # Check that the wave in each scope channel contains the expected number of samples.
            assert (len(wave) == totalsamples), f"Scope record {index}/{num_records} size does not match totalsamples."

def get_scope_records(scope_module, num_records: int):
    # Obtain scope records from the device using an instance of the Scope Module.
    scope_module.raw_module.execute()
    uhf.scopes[0].enable(True)
    session.sync()
    start = time.time()
    timeout = 100000 # [s]
    records = 0
    progress = 0
    # Wait until the Scope Module has received and processed the desired number of records.
    while (records < num_records): # or (progress < 1.0):
        time.sleep(0.1)
        records = scope_module.records()
        progress = scope_module.raw_module.progress()[0]
        print(f"Scope module has acquired {records} records (requested {num_records}). Progress of current segment {100.0 * progress}%.",end="\r")
        if (time.time() - start) > timeout:
            # Break out of the loop if for some reason we're no longer receiving scope data from the device.
            print(f"\nScope Module did not return {num_records} records after {timeout} s - forcing stop.")
            break
    uhf.scopes[0].enable(True)
    # Read out the scope data from the module.
    data = scope_module.raw_module.read(True)[wave_node.node_info.path]
    # Stop the module; to use it again we need to call execute().
    scope_module.raw_module.finish()
    check_scope_record_flags(data, num_records)
    return data

def to_timestamp(record):
    totalsamples = record[0]["totalsamples"]
    dt = record[0]["dt"]
    timestamp = record[0]["timestamp"]
    triggertimestamp = record[0]["triggertimestamp"]
    t = (np.arange(-totalsamples, 0)*dt) + ((timestamp - triggertimestamp)/float(clockbase))
    return t

def to_frequency(record, scope_time):
    totalsamples = record[0]["totalsamples"]
    scope_rate = clockbase / 2 ** scope_time
    return np.linspace(0, scope_rate / 2, totalsamples)